# Train AI Text Detector

This notebook uses the existing project modules to load data, create a train/test split, train the detector, and save everything needed for a later evaluation notebook run.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from dataclasses import asdict
from pathlib import Path

from sklearn.model_selection import train_test_split

from dataset_loader import DatasetConfig, load_combined_dataset
from train import TrainConfig, train


In [ ]:
def split_loaded_data(result, test_ratio: float, seed: int):
    indices = list(range(len(result.texts)))
    stratify = None

    source_label_groups = [
        f"{result.samples[i].source_dataset}:{result.labels[i]}"
        for i in indices
    ]
    group_counts = Counter(source_label_groups)
    if group_counts and min(group_counts.values()) >= 2:
        stratify = source_label_groups
    else:
        label_counts = Counter(result.labels)
        if label_counts and min(label_counts.values()) >= 2:
            stratify = result.labels

    try:
        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_ratio,
            random_state=seed,
            stratify=stratify,
        )
    except ValueError:
        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_ratio,
            random_state=seed,
            stratify=None,
        )

    train_texts = [result.texts[i] for i in train_idx]
    train_labels = [result.labels[i] for i in train_idx]
    test_texts = [result.texts[i] for i in test_idx]
    test_labels = [result.labels[i] for i in test_idx]
    test_samples = [result.samples[i] for i in test_idx]
    return train_texts, train_labels, test_texts, test_labels, test_samples


artifact_dir = Path("checkpoints/notebook_run")
test_ratio = 0.15

data_config = DatasetConfig(
    sources=["hc3", "gpt_wiki"],
    max_per_source=1000,
    max_total=2000,
    min_text_length=35,
    max_text_length=3000,
    balance_labels=True,
    seed=42,
    cache_dir=".cache/datasets",
    cache_sources=True,
)

train_config = TrainConfig(
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_feedforward=512,
    max_len=512,
    epochs=5,
    batch_size=32,
    lr=3e-4,
    max_vocab=50_000,
    seed=data_config.seed,
    save_dir=str(artifact_dir),
)

artifact_dir


In [ ]:
result = load_combined_dataset(data_config)
train_texts, train_labels, test_texts, test_labels, test_samples = split_loaded_data(
    result,
    test_ratio=test_ratio,
    seed=data_config.seed,
)

print(f"Total samples: {len(result.texts):,}")
print(f"Train samples: {len(train_texts):,}")
print(f"Test samples: {len(test_texts):,}")
print("Train labels:", Counter(train_labels))
print("Test labels:", Counter(test_labels))
print("Test sources:", Counter(sample.source_dataset for sample in test_samples))


In [ ]:
model, tokenizer, threshold = train(train_texts, train_labels, train_config)

artifact_dir.mkdir(parents=True, exist_ok=True)
with (artifact_dir / "test_samples.jsonl").open("w", encoding="utf-8") as handle:
    for sample in test_samples:
        handle.write(
            json.dumps(
                {
                    "text": sample.text,
                    "label": int(sample.label),
                    "source_dataset": sample.source_dataset,
                    "generator": sample.generator,
                    "domain": sample.domain,
                },
                ensure_ascii=False,
            )
            + "\n"
        )

run_config = {
    "artifact_dir": str(artifact_dir),
    "test_ratio": test_ratio,
    "data_config": asdict(data_config),
    "train_config": asdict(train_config),
}
(artifact_dir / "notebook_run_config.json").write_text(
    json.dumps(run_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Saved checkpoint directory: {artifact_dir}")
print(f"Saved test split: {artifact_dir / 'test_samples.jsonl'}")
print(f"Saved notebook config: {artifact_dir / 'notebook_run_config.json'}")
print(f"Calibrated threshold: {threshold:.3f}")
